# Bot Discord Colab Runner

Notebook para executar o projeto no Google Colab.

Use este notebook como launcher. O codigo real fica no GitHub. Para testar novas fases, atualize o repositorio no GitHub e rode a celula de `git pull`.

## 1. Configurar repositorio

Troque `REPO_URL` pela URL do seu repositorio no GitHub. Se o repo for privado, use uma URL/token de acesso ou clone manualmente pelo Colab.

In [ ]:
REPO_URL = "https://github.com/OtaoriGz/bot-discord-colab.git"
BRANCH = "main"
REPO_DIR = "/content/bot-discord-colab"
SRC_DIR = f"{REPO_DIR}/src"

print("Repo:", REPO_URL)
print("Branch:", BRANCH)
print("Diretorio:", REPO_DIR)

## 2. Verificar runtime

Para fases com STT/TTS, use GPU. Para a fase atual de Discord voice, CPU ja consegue rodar.

In [ ]:
import sys

print("Python:", sys.version)

try:
    import torch
    print("CUDA disponivel:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch nao carregou ou nao esta instalado:", exc)

## 3. Instalar dependencias de sistema

`ffmpeg` e necessario para tocar audio no Discord.

In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg git

## 4. Clonar ou atualizar o repo

Rode esta celula sempre que atualizar o GitHub.

In [ ]:
import os

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git status --short

## 5. Instalar dependencias Python

In [ ]:
%cd {REPO_DIR}
!python -m pip install -U pip
!pip install -r requirements.txt

## 6. Configurar variaveis secretas

O token nao fica salvo no repo. Ele fica so no runtime do Colab.

In [ ]:
import os
from getpass import getpass

if not os.getenv("DISCORD_TOKEN"):
    os.environ["DISCORD_TOKEN"] = getpass("Discord bot token: ")

# Opcional por enquanto. Sera usado nas proximas fases.
if not os.getenv("LLM_API_KEY"):
    value = getpass("Groq/OpenRouter API key opcional (Enter para pular): ")
    if value:
        os.environ["LLM_API_KEY"] = value

# Padrao: Groq. Para OpenRouter, troque para https://openrouter.ai/api/v1 e escolha um modelo OpenRouter.
os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
os.environ.setdefault("LLM_MODEL", "llama-3.1-8b-instant")

print("DISCORD_TOKEN configurado:", bool(os.getenv("DISCORD_TOKEN")))
print("LLM_API_KEY configurado:", bool(os.getenv("LLM_API_KEY")))
print("LLM_BASE_URL:", os.getenv("LLM_BASE_URL"))
print("LLM_MODEL:", os.getenv("LLM_MODEL"))

## 7. Configurar canal e arquivos opcionais

Se voce nao preencher `BOT_ACTIVE_VOICE_CHANNEL_ID`, o comando `/join` entra na call em que voce estiver.

In [ ]:
# Opcional: fixar servidor/call/membros escutados sem editar YAML.
# os.environ["BOT_ACTIVE_GUILD_ID"] = "123456789012345678"
# os.environ["BOT_ACTIVE_VOICE_CHANNEL_ID"] = "123456789012345678"
# os.environ["BOT_LISTEN_FILTER"] = "111111111111111111,222222222222222222"

os.environ["TEST_AUDIO_PATH"] = "/content/test_audio.wav"
os.environ["DISCORD_RECORDINGS_DIR"] = "/content/discord_recordings"

print("TEST_AUDIO_PATH:", os.environ["TEST_AUDIO_PATH"])
print("DISCORD_RECORDINGS_DIR:", os.environ["DISCORD_RECORDINGS_DIR"])

## 8. Adicionar src ao Python path e validar import

In [ ]:
import sys

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from bot_discord_colab.main import run, run_async
from bot_discord_colab.config import load_config

config = load_config()
print("Import OK")
print("Bot:", config.name)
print("Idioma:", config.language)
print("TTS:", config.tts_model)
print("LLM:", config.llm_model)

## 9. Rodar modo sem Discord

Esta celula confirma que o projeto carrega sem conectar no Discord.

In [ ]:
runtime = run(start_discord=False)
runtime.keys()

## 10. Criar audio de teste

Esse arquivo sera usado pelo comando `/play_test_audio`.

In [ ]:
!ffmpeg -y -f lavfi -i sine=frequency=440:duration=2 -ar 48000 -ac 2 /content/test_audio.wav

from IPython.display import Audio, display
display(Audio("/content/test_audio.wav"))

## 11. Iniciar bot Discord

Esta celula fica rodando. Depois que aparecer que o bot esta online, va no Discord e use:

- `/status`
- `/join`
- `/play_test_audio path:/content/test_audio.wav`
- `/record_test seconds:5`
- `/leave`

Para parar, interrompa a execucao da celula.

In [ ]:
from bot_discord_colab.main import run_async

await run_async()

## 12. Ver arquivos gravados

Rode depois de usar `/record_test`.

In [ ]:
!find /content/discord_recordings -maxdepth 1 -type f -print || true

## 13. Atualizar projeto em fases futuras

Quando voce subir nova fase no GitHub, rode de novo:

1. Celula 4: clonar/atualizar repo.
2. Celula 5: instalar dependencias.
3. Celula 8: validar imports.
4. Celula 11: iniciar bot.